# Inspect and Improve Convergence

This guide covers how to read convergence metrics, visualise area errors and generator displacement, and tune parameters to reach a satisfactory result faster.

See also: [Tutorial](../../tutorials/basic-voronoi-cartogram.ipynb) · [Explanation: CVT Algorithm](../../explanations/voronoi-cartogram-algorithm.md) · [Analyze and Fix Topology](topology.ipynb)

In [ ]:
import matplotlib.pyplot as plt

import carto_flow.data as examples
import carto_flow.voronoi_cartogram as vor
from carto_flow.voronoi_cartogram.visualization import plot_convergence

us_states = examples.load_us_census(population=True)

## Run With History and Progress Logging

Pass `show_progress=True` to print per-iteration diagnostics and `record_history=5` to snapshot every 5 iterations.

In [ ]:
result = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
    options=vor.VoronoiOptions(
        n_iter=30,
        show_progress=True,
        record_history=5,
    ),
)

## Read the Metrics

The `metrics` dict is the first place to look after a run.

| Key | Meaning |
|---|---|
| `converged` | `True` if a tolerance was set and reached |
| `n_iterations` | Total iterations run |
| `initial_area_cv` | Area coefficient of variation before relaxation |
| `final_area_cv` | Area CV after relaxation; lower is better |
| `mean_area_error_pct` | Average absolute % deviation from target area |
| `max_area_error_pct` | Worst-case % deviation |

In [ ]:
for k, v in result.metrics.items():
    print(f"  {k:<25} {v}")

## Plot the Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
plot_convergence(result, ax=ax)
ax.set_ylim(0, None)
plt.tight_layout();

A still-decreasing curve means more iterations will help. A flat curve means the algorithm has stalled and parameter changes are needed.

## Per-Region Area Errors

`area_errors` gives the signed % deviation from each region's target area.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
result.plot(
    column="area_error_pct",
    ax=ax,
    cmap="RdYlGn_r",
    vmin=-70,
    vmax=70,
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "Area error (%)"},
)
for cell, label in zip(result.cells, us_states["State Abbreviation"], strict=False):
    ax.text(cell.centroid.x, cell.centroid.y, label, fontsize=7, ha="center", va="center")
ax.set(title="Per-region area error")
ax.axis("off")
plt.tight_layout();

## Displacement Plot

`plot_displacement()` draws arrows from each region's original centroid to its final generator position, showing which regions moved most during relaxation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
result.plot_displacement(ax=ax, show_adjacency=False)
ax.set(title="Generator displacement")
ax.axis("off")
plt.tight_layout();

## Improving Convergence

### More iterations

Increase `n_iter`. Set `area_cv_tol` to stop automatically once a target is reached (typical acceptable range for US states: 0.05–0.10).

In [ ]:
result_more = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
    options=vor.VoronoiOptions(n_iter=100, area_cv_tol=0.05),
)
print(
    f"Converged: {result_more.metrics['converged']}  "
    f"after {result_more.metrics['n_iterations']} iterations  "
    f"(area CV: {result_more.metrics['final_area_cv']:.4f})"
)

### Over-relaxation schedule

The default `"overrelax"` schedule applies a decaying over-relaxation factor that overshoots each step slightly and anneals toward standard Lloyd. You can customise `start`, `decay`, and `minimum`.

In [ ]:
# Default equivalent — shown explicitly for reference
# schedule = vor.RelaxationSchedule(start=1.9, decay=0.98, minimum=1.0)

# slower decay of over-relaxation
schedule = vor.RelaxationSchedule(start=1.9, decay=0.999, minimum=1.0)

result_relax = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
    backend=vor.RasterBackend(relaxation=schedule),
    options=vor.VoronoiOptions(n_iter=30),
)
print(f"Final area CV: {result_relax.metrics['final_area_cv']:.4f}")

### Higher resolution

A larger pixel grid (`RasterBackend(resolution=500)`) yields smoother cells and more accurate centroids, at the cost of longer iteration times. Convergence may improve if the initial resolution was too low for the number of input geometries.

In [ ]:
result_hires = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
    backend=vor.RasterBackend(resolution=500),
    options=vor.VoronoiOptions(n_iter=30),
)
print(f"Final area CV: {result_hires.metrics['final_area_cv']:.4f}")

### Elastic boundaries

By making the outer boundary elastic and move to accomodate shtinking/growing cells, convergence can usually be improved considerably.

In [ ]:
result_elastic = vor.create_voronoi_cartogram(
    us_states,
    weights="Population (Millions)",
    backend=vor.RasterBackend(boundary=vor.ElasticBoundary(strength=0.05)),
    options=vor.VoronoiOptions(n_iter=30),
)
print(f"Final area CV: {result_elastic.metrics['final_area_cv']:.4f}")

## See Also

- [Tutorial](../../tutorials/basic-voronoi-cartogram.ipynb)
- [Explanation: CVT Algorithm](../../explanations/voronoi-cartogram-algorithm.md)
- [Analyze and Fix Topology](topology.ipynb) — `plot_displacement` also helps diagnose topology issues